# 07 · Explicabilidad (SHAP), calibración y zonas de decisión

**Qué hace este notebook.** (1) Reconstruye el modelo con los hiperparámetros del notebook 06; (2) fija las **3 zonas de decisión** solo con predicciones fuera de pliegue (OOF) del entrenamiento; (3) mira la reserva **solo para informar** (no decide nada); (4) explica el modelo con SHAP; (5) congela el artefacto `models/model.joblib`.

> **Aviso de método (regla 4).** La reserva ya se abrió dos veces. Esta es una *tercera mirada de solo lectura*: los umbrales `T_auto` y `T_low` salen del entrenamiento y no se tocan después de ver la reserva. Si el resultado desagrada, no se reajusta nada con ella.

In [1]:
import sys, json, warnings, datetime
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve(); ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, joblib
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, shap
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from config.settings import PROCESSED_DIR, REPORTS_DIR, ROOT as PROJECT_ROOT, SEED, TARGET, GROUP_COL, AUTO_APPROVE_CONFIDENCE
from accessguard.engineering import IAMFeatureBuilder
from accessguard.modeling import build_model, fold_matrices, make_cv, metrics_at
from accessguard.security_metrics import (auto_approval_at_zero_fpr, auto_approval_at_confidence, queue_capture, fpr_upper_bound)
from accessguard.explain import (assign_zone, compute_t_low, zone_summary, make_explainer, shap_values, scaled_for_plot,
                                 impact_table, expected_calibration_error, ZONE_LABELS)
from sklearn.pipeline import Pipeline
MODELS_DIR = PROJECT_ROOT / "models"; FIG_DIR = REPORTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True); MODELS_DIR.mkdir(exist_ok=True)

best = json.loads((REPORTS_DIR / "06_zt_best_params.json").read_text(encoding="utf-8"))
MODEL, HP, T_AUTO_06 = best["model"], best["best_params"], best["t_auto_strict"]
print("Modelo:", MODEL, "· hiperparámetros del notebook 06:", {k: round(v, 4) for k, v in HP.items()})

Modelo: LogReg ElasticNet · hiperparámetros del notebook 06: {'C': 0.0317, 'l1_ratio': 0.3393}


## 1. Modelo con los hiperparámetros del notebook 06
Se reentrena con todo el entrenamiento (misma semilla) y se comprueba que da lo mismo que el modelo exportado en el notebook 06.

In [2]:
train = pd.read_csv(PROCESSED_DIR / "train_clean.csv")
hold = pd.read_csv(PROCESSED_DIR / "holdout_clean.csv")
y = train[TARGET].to_numpy(); X = train.drop(columns=TARGET)
yh = hold[TARGET].to_numpy(); Xh = hold.drop(columns=TARGET)
assert not set(Xh[GROUP_COL]) & set(X[GROUP_COL]), "hay gerentes de la reserva en el entrenamiento"

fb = IAMFeatureBuilder(seed=SEED)
X_feat = fb.fit_transform(X, y)                       # target encoding fuera de pliegue (sin fuga)
model = build_model(MODEL, HP, seed=SEED).fit(X_feat, y)
pipe = Pipeline([("features", fb), ("model", model)])

old = joblib.load(MODELS_DIR / "model.joblib")
p_h = pipe.predict_proba(Xh)[:, 1]
diff = float(np.abs(p_h - old["pipeline"].predict_proba(Xh)[:, 1]).max())
print(f"{len(X):,} filas de entrenamiento · {len(Xh):,} de reserva · {X_feat.shape[1]} variables · diferencia máx. con el modelo del notebook 06: {diff:.2e}")

26,215 filas de entrenamiento · 6,554 de reserva · 29 variables · diferencia máx. con el modelo del notebook 06: 0.00e+00


## 2. Umbrales y zonas de decisión (solo con el entrenamiento)

| Zona | Regla | Qué pasa |
|---|---|---|
| **Verde** | p > `T_auto` (umbral estricto: FPR = 0 % en validación cruzada) | Auto-aprobación |
| **Amarillo** | `T_low` ≤ p ≤ `T_auto` | Cola de triaje: un analista IAM revisa, de más a menos sospechosa |
| **Rojo** | p < `T_low` (el 10 % más sospechoso) | Revisión manual a fondo; nunca se auto-deniega |

`T_low` es una decisión de capacidad del equipo IAM (aquí, 10 % de las solicitudes); se puede cambiar sin volver a entrenar.

In [3]:
folds = fold_matrices(X, y, X[GROUP_COL], make_cv(5, SEED), seed=SEED)
p_oof = np.zeros(len(y))
for X_tr, y_tr, X_va, y_va, idx_va in folds:
    p_oof[idx_va] = build_model(MODEL, HP, seed=SEED).fit(X_tr, y_tr).predict_proba(X_va)[:, 1]

z = auto_approval_at_zero_fpr(y, p_oof)
T_AUTO = z["threshold"]; RED_FRACTION = 0.10
T_LOW = compute_t_low(p_oof, RED_FRACTION)
assert np.isclose(T_AUTO, T_AUTO_06), "el umbral estricto no coincide con el del notebook 06"
print(f"T_auto = {T_AUTO:.4f} (igual al notebook 06) · T_low = {T_LOW:.4f} (percentil {RED_FRACTION:.0%} de p en OOF)")
oof_zones = zone_summary(y, p_oof, T_LOW, T_AUTO)
oof_zones.round(4)

T_auto = 0.9809 (igual al notebook 06) · T_low = 0.7450 (percentil 10% de p en OOF)


,decisión,n,% del total,accesos indebidos,% indebidos en la zona,lift vs base
zona,,,,,,
verde,Auto-aprobación estricta,893,0.0341,0,0.0000,0.0000
amarillo,Cola de triaje priorizada,22700,0.8659,1023,0.0451,0.7783
rojo,Revisión manual,2622,0.1000,495,0.1888,3.2602


## 3. Calibración
¿Una `p` de 0,95 significa que ~95 % se aprueban de verdad? Se mide en OOF (entrenamiento) y en la reserva. **No se recalibra:** hacerlo movería los umbrales ya fijados.

In [4]:
cal = {}
fig, ax = plt.subplots(figsize=(5, 4))
for name, yy, pp, col in [("OOF (entrenamiento)", y, p_oof, "#2A6FBB"), ("Reserva", yh, p_h, "#D9822B")]:
    frac, mean_p = calibration_curve(yy, pp, n_bins=10, strategy="quantile")
    ax.plot(mean_p, frac, "o-", color=col, label=name)
    cal[name] = {"brier": float(brier_score_loss(yy, pp)), "ece": expected_calibration_error(yy, pp)}
ax.plot([0.5, 1], [0.5, 1], "k--", lw=1, label="perfecta"); ax.set_xlabel("p(aprobar) media"); ax.set_ylabel("fracción realmente aprobada")
ax.legend(); ax.set_title("Curva de calibración"); plt.tight_layout(); plt.savefig(FIG_DIR / "calibracion.png", dpi=130); plt.close()
pd.DataFrame(cal).T.round(4)

,brier,ece
OOF (entrenamiento),0.0608,0.0812
Reserva,0.0612,0.0786


## 4. Reserva (solo lectura): rendimiento y zonas

In [5]:
m_h = metrics_at(yh, p_h)
zones_h = zone_summary(yh, p_h, T_LOW, T_AUTO)
strict_fp = int(((p_h > T_AUTO) & (yh == 0)).sum()); n_neg_h = int((yh == 0).sum())
prac = auto_approval_at_confidence(yh, p_h, AUTO_APPROVE_CONFIDENCE)
cap = queue_capture(yh, p_h, T_AUTO, fractions=(0.05, 0.10, 0.20, 0.50))
print(f"Reserva: ROC-AUC = {m_h['roc_auc']:.4f} · PR-AUC denegar = {m_h['pr_auc_deny']:.4f} (azar = {1 - yh.mean():.4f})")
print(f"Estricto: {int((p_h > T_AUTO).sum())} auto-aprobadas, {strict_fp} accesos indebidos (FPR {strict_fp / n_neg_h:.2%}, real ≤ {fpr_upper_bound(strict_fp, n_neg_h):.2%})")
print("Captura de denegaciones en la cola ordenada:", {k: round(v, 3) for k, v in cap.items() if k != 'n_queue'})
zones_h.round(4)

Reserva: ROC-AUC = 0.7009 · PR-AUC denegar = 0.1987 (azar = 0.0578)
Estricto: 341 auto-aprobadas, 2 accesos indebidos (FPR 0.53%, real ≤ 1.65%)
Captura de denegaciones en la cola ordenada: {'5%': 0.208, '10%': 0.306, '20%': 0.417, '50%': 0.728}


,decisión,n,% del total,accesos indebidos,% indebidos en la zona,lift vs base
zona,,,,,,
verde,Auto-aprobación estricta,341,0.0520,2,0.0059,0.1014
amarillo,Cola de triaje priorizada,5546,0.8462,261,0.0471,0.8138
rojo,Revisión manual,667,0.1018,116,0.1739,3.0075


## 5. SHAP: qué empuja la decisión
Valores SHAP del modelo final en el espacio log-odds de **aprobar** (positivo = empuja a aprobar). Se calculan sobre las variables de la reserva; el fondo de referencia son 300 filas de entrenamiento.

In [6]:
rng = np.random.default_rng(SEED)
background = X_feat.iloc[rng.choice(len(X_feat), 300, replace=False)].reset_index(drop=True)
explainer = make_explainer(pipe, background)
H_feat = fb.transform(Xh)
sv = shap_values(pipe, explainer, H_feat)

plt.figure()
shap.summary_plot(sv.to_numpy(), scaled_for_plot(pipe, H_feat), feature_names=list(H_feat.columns), max_display=15, show=False)
plt.title("SHAP · impacto en la probabilidad de aprobar (reserva)")
plt.tight_layout(); plt.savefig(FIG_DIR / "shap_summary.png", dpi=140, bbox_inches="tight"); plt.close()

imp = impact_table(sv)
imp.head(12).round(4)

Background dataset has 300 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=300 when initializing the masker.


,mean_abs_shap,mean_shap,share
freq__RESOURCE_x_MGR_ID,1.0624,-1.0624,0.2622
freq__RESOURCE_x_ROLE_DEPTNAME,0.7614,-0.4230,0.1879
freq__MGR_ID_x_ROLE_TITLE,0.5811,0.5811,0.1434
freq__RESOURCE,0.1771,0.0350,0.0437
te__RESOURCE,0.1592,-0.0367,0.0393
nuniq__ROLE_DEPTNAME_por_RESOURCE,0.1315,-0.0045,0.0325
freq__ROLE_ROLLUP_2,0.1268,-0.0285,0.0313
freq__ROLE_TITLE_x_ROLE_DEPTNAME,0.1231,-0.0629,0.0304
te__ROLE_DEPTNAME,0.1222,-0.0092,0.0302
te__ROLE_ROLLUP_2,0.1194,-0.0178,0.0295


In [7]:
# aditividad: base + suma SHAP = logit(p) (comprobación de que la explicación es fiel al modelo)
logit = np.log(p_h / (1 - p_h))
recon = float(np.ravel(explainer.expected_value)[-1]) + sv.sum(axis=1).to_numpy()
print("error máx. de reconstrucción del logit:", float(np.abs(logit - recon).max()))

families = {"te__": "target encoding", "freq__": "frecuencia", "nuniq__": "nº de distintos"}
fam = {v: float(imp.loc[[i for i in imp.index if i.startswith(k)], "share"].sum()) for k, v in families.items()}
fam["otras"] = float(1 - sum(fam.values()))
print("Peso por familia de variables:", {k: round(v, 3) for k, v in fam.items()})

error máx. de reconstrucción del logit: 3.019806626980426e-14
Peso por familia de variables: {'target encoding': 0.192, 'frecuencia': 0.742, 'nº de distintos': 0.057, 'otras': 0.009}


## 6. Artefactos: reporte JSON y modelo congelado

In [8]:
report = {
    "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "model": MODEL, "best_params": HP,
    "method_note": "Tercera lectura de la reserva, solo informativa: umbrales fijados con OOF del entrenamiento.",
    "thresholds": {"t_auto_strict": T_AUTO, "t_low": T_LOW, "red_fraction": RED_FRACTION, "practical_confidence": AUTO_APPROVE_CONFIDENCE},
    "oof_zones": oof_zones.reset_index().to_dict(orient="records"),
    "holdout": {"n": int(len(yh)), "roc_auc": m_h["roc_auc"], "pr_auc_deny": m_h["pr_auc_deny"],
                "strict_fp": strict_fp, "strict_fpr": strict_fp / n_neg_h, "strict_fpr_upper95": fpr_upper_bound(strict_fp, n_neg_h),
                "practical_098": prac, "queue_capture": cap, "zones": zones_h.reset_index().to_dict(orient="records")},
    "calibration": cal,
    "shap": {"n_explained": int(len(sv)), "explainer": type(explainer).__name__, "family_share": fam,
             "top_features": [{"feature": k, "mean_abs_shap": float(r.mean_abs_shap), "mean_shap": float(r.mean_shap), "share": float(r.share)}
                              for k, r in imp.head(15).iterrows()]},
}
(REPORTS_DIR / "07_shap_analysis.json").write_text(json.dumps(report, indent=2, ensure_ascii=False, default=float), encoding="utf-8")

bundle = dict(old)                       # se conservan las claves del notebook 06
bundle.update({
    "pipeline": pipe, "model_name": MODEL, "best_params": HP,
    "t_auto_strict": float(T_AUTO), "t_low": float(T_LOW),
    "zones": {"verde": "p > t_auto_strict: auto-aprobación estricta", "amarillo": "t_low <= p <= t_auto_strict: cola de triaje priorizada",
              "rojo": "p < t_low: revisión manual", "labels": ZONE_LABELS},
    "shap_background": background, "feature_names": list(X_feat.columns),
    "holdout_metrics": {"roc_auc": m_h["roc_auc"], "pr_auc_deny": m_h["pr_auc_deny"], "strict_fp": strict_fp},
    "frozen_at": report["generated_at"],
})
joblib.dump(bundle, MODELS_DIR / "model.joblib")
chk = joblib.load(MODELS_DIR / "model.joblib")
assert np.allclose(chk["pipeline"].predict_proba(Xh.head(50))[:, 1], p_h[:50])
print("models/model.joblib congelado:", round((MODELS_DIR / "model.joblib").stat().st_size / 1024), "KB · claves:", sorted(chk))

models/model.joblib congelado: 8671 KB · claves: ['best_params', 'cv_metrics_oof', 'decision_rule', 'feature_names', 'framework', 'frozen_at', 'holdout_metrics', 'input_columns', 'latency_ms', 'model_name', 'pipeline', 'shap_background', 't_auto_strict', 't_confidence', 't_low', 'zones']


## 7. Qué concluir (y qué no)
* Las zonas se fijaron con el entrenamiento; la tabla de la reserva muestra si **se sostienen** en gerentes nunca vistos.
* SHAP explica **cómo decide el modelo**, no *por qué* un gerente aprueba: los códigos son anónimos y el target encoding resume el historial de aprobaciones, así que las variables `te__*` dicen "históricamente se aprobó mucho en esta combinación".
* La calibración se reporta pero no se corrige; una recalibración exigiría volver a fijar `T_auto` con OOF.
* Cualquier cifra con 0 FP observados lleva su cota superior al 95 %.